# 📘 Notebook 15: Subword Tokenisation and Decoding

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module F: Language Models in Practice · Notebook 1 of 4 in this module · (15 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why modern language models use neither words nor characters as their units, but **subwords**
- Implement **byte-pair encoding (BPE)** by hand: learn the merges and apply them to unseen words
- Inspect how a real model's tokeniser splits text, and explain why the number of tokens matters
- Describe a language model's output as a **probability distribution** over the next token
- Implement **greedy** decoding, **temperature**, **top-k** and **top-p** sampling, and **beam search**
- Choose a decoding strategy to suit a task

> **Prerequisites:** Notebooks 03, 07 and 12 of this course. The Transformer and pretrained models are covered in *From Python to Deep Learning & Fine-Tuning*, Notebooks 16 and 17. This notebook treats a large language model as a given, and studies its two ends: how text gets **in**, and how text comes **out**.
>
> 🔭 **Why this notebook matters:** between your prompt and a model's reply lie two steps that are easy to overlook and that shape everything you see. Before the model runs, your text is cut into **tokens**, which is what you are billed for and what limits how much the model can read. After the model runs, it has produced only probabilities, and a separate procedure must turn them into words. The settings called *temperature* and *top-p* in every AI product belong to that procedure. After this notebook you will know exactly what they do.

> ℹ️ **Setup note.** Run this cell once. Most of the notebook runs on plain Python. Two short sections near the end download the small GPT-2 model and are marked as such.

> 📦 **Libraries used in this notebook.**
>
> - **NumPy**, **Matplotlib** and **NLTK** only. The tokeniser and all the decoding methods are written by hand.
> - Sections 3 and 6 use **Transformers** (Notebook 13) to load the tokeniser and the weights of GPT-2. Its installation line is in the cell that uses it.
>
> The cell below installs what is needed. In Google Colab these libraries are already present and the cell finishes in a moment. On your own computer it may take a minute the first time.

In [ ]:
%pip install -q nltk numpy matplotlib

In [ ]:
import nltk
for package in ["gutenberg", "punkt", "punkt_tab"]:
    nltk.download(package, quiet=True)

import math
import re
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

## 1. The vocabulary dilemma

### Intuition first
A neural language model has a fixed list of units it can read and write, its **vocabulary**. What should the units be?

**Words** are the obvious choice, and Notebook 01 already told us what goes wrong. By Zipf's law, most words are rare. Whatever list of words you fix, real text will keep producing words that are not on it: names, new coinages, technical terms, typing errors, and all the forms of a word you happened not to see. A word-level model can only replace them with a meaningless `<unk>`.

Let us measure the problem. We build a vocabulary from one novel and count how much of **another** novel it fails to cover.

In [ ]:
from nltk.corpus import gutenberg

def words_of(file_id):
    return re.findall(r"[a-z]+", gutenberg.raw(file_id).lower())

emma_words = words_of("austen-emma.txt")
alice_words = words_of("carroll-alice.txt")
known = set(emma_words)

unknown_tokens = [word for word in alice_words if word not in known]
unknown_types = set(unknown_tokens)

print(f"Vocabulary learned from Emma: {len(known):,} words")
print(f"Alice: {len(unknown_tokens) / len(alice_words):.1%} of the tokens and {len(unknown_types) / len(set(alice_words)):.0%} of the word types are unknown")
print(sorted(unknown_types, key=len, reverse=True)[:12])

Both books are nineteenth-century English novels, and a vocabulary of several thousand words still misses a sizeable share of the second book's vocabulary. Look at what is missing: many are ordinary words such as plurals and verb forms whose **parts** are perfectly familiar.

**Characters** are the opposite choice, and we used them in Notebook 12. Nothing is ever unknown, since everything is spelled with the same few dozen symbols. The price is that sequences become five or six times longer, and that the model must learn from scratch that `c-a-t` means something.

**Subwords** are the compromise that every modern model uses. Frequent words are kept whole. Rare words are split into smaller, meaningful, reusable pieces: *unbelievable* becomes something like *un* + *believ* + *able*. The vocabulary stays small, nothing is ever unknown, and the pieces carry meaning.

The remaining question is how to choose the pieces. The answer should not surprise you by now: let the data decide.

## 2. Byte-pair encoding by hand

### Intuition first
**Byte-pair encoding (BPE)** started life in 1994 as a method for compressing files, and was adapted for language in 2016. It is very simple.

1. Start with every word split into single characters.
2. Find the pair of neighbouring symbols that occurs **most often** in the whole corpus.
3. **Merge** that pair into a new single symbol, everywhere.
4. Repeat, for as many merges as you want.

Frequent sequences of letters fuse together step by step: first common pairs such as `t`+`h`, then common endings such as `ing`, and eventually whole frequent words. The list of merges, in the order they were made, **is** the tokeniser.

### A toy example
We mark the end of each word with the symbol `</w>`, so that the model can tell *est* at the end of *newest* from *est* at the start of *estate*.

In [ ]:
def merge_pair(symbols, pair):
    """Replace every occurrence of the pair in a tuple of symbols by one merged symbol."""
    merged = []
    i = 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            merged.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            merged.append(symbols[i])
            i += 1
    return tuple(merged)

def learn_bpe(word_counts, number_of_merges, show=False):
    words = {tuple(word) + ("</w>",): count for word, count in word_counts.items()}
    merges = []
    for step in range(number_of_merges):
        pair_counts = Counter()
        for symbols, count in words.items():
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += count
        if not pair_counts:
            break
        best = max(pair_counts, key=pair_counts.get)
        merges.append(best)
        words = {merge_pair(symbols, best): count for symbols, count in words.items()}
        if show:
            print(f"merge {step + 1}: {best[0]} + {best[1]}  (seen {pair_counts[best]} times)   {[' '.join(s) for s in words]}")
    return merges

toy_counts = Counter("low low low low low lower lower newest newest newest newest newest newest widest widest widest".split())
print(dict(toy_counts), "\n")

toy_merges = learn_bpe(toy_counts, 8, show=True)

Follow the merges. The most frequent pair is fused first, then the next, and familiar units take shape: the ending *est* with its end-of-word marker, and the stem *low*. Nobody told the algorithm that *-est* is an English suffix. It is simply a frequent sequence.

### Applying the merges to a new word
To tokenise any word, split it into characters and replay the merges in the order they were learned.

In [ ]:
def bpe_tokenise(word, merges):
    symbols = tuple(word) + ("</w>",)
    for pair in merges:
        symbols = merge_pair(symbols, pair)
    return list(symbols)

for word in ["lowest", "newer", "wide"]:
    print(f"{word:<8} {bpe_tokenise(word, toy_merges)}")

None of these three words was in the toy corpus. *lowest* is nevertheless split into two pieces the tokeniser knows well, *low* and *est*. *newer* keeps the familiar stem *new*, and *wide*, which shares nothing with the corpus, is spelled out letter by letter. A model that has learned what *low* means and what *-est* does has a fair chance with *lowest*, although it has never seen it. This is the whole point of subwords.

### On a real book
Now we learn 1,000 merges from the words of *Emma*.

In [ ]:
emma_counts = Counter(emma_words)
merges = learn_bpe(emma_counts, 1000)

print("First 12 merges:", [a + "+" + b for a, b in merges[:12]])
print("Last 6 merges:  ", [a + "+" + b for a, b in merges[-6:]])
print()
for word in ["the", "happiness", "unhappily", "caterpillar", "tokenisation", "wonderlandish", "xylophone"]:
    print(f"{word:<14} {bpe_tokenise(word, merges)}")

The first merges build the commonest letter pairs and the commonest short words. Later ones build longer pieces. Frequent words such as *the* have become single tokens. A word the tokeniser never saw is broken into several pieces, down to single letters where necessary, but **it is never unknown**.

How does the trade-off look in numbers? We tokenise the whole of *Alice*, a book the tokeniser has never seen, in three ways.

In [ ]:
cache = {}
def tokens_of(word):
    if word not in cache:
        cache[word] = bpe_tokenise(word, merges)
    return cache[word]

as_words = len(alice_words)
as_characters = sum(len(word) + 1 for word in alice_words)
as_subwords = sum(len(tokens_of(word)) for word in alice_words)

symbols_in_use = set(symbol for word in emma_counts for symbol in tokens_of(word))

print(f"{'units':<12} {'vocabulary size':>16} {'tokens for Alice':>18} {'unknown tokens':>16}")
print(f"{'words':<12} {len(known):>16,} {as_words:>18,} {len(unknown_tokens) / len(alice_words):>15.1%}")
print(f"{'subwords':<12} {len(symbols_in_use):>16,} {as_subwords:>18,} {0:>15.1%}")
print(f"{'characters':<12} {27:>16,} {as_characters:>18,} {0:>15.1%}")

The subword vocabulary is a fraction of the size of the word vocabulary, it leaves nothing unknown, and it needs about one and a half times as many tokens as whole words, against five times as many for characters. That is why it won.

> 🧠 The number of merges is the one setting of BPE, and it fixes the vocabulary size. Real models use between about 30,000 and 200,000 pieces. Other subword methods (WordPiece, used by BERT, and SentencePiece) differ in detail, and the idea is the same.

## 3. A real tokeniser

*This section downloads the tokeniser of GPT-2, a few megabytes.*

GPT-2 uses BPE with about 50,000 pieces, learned from a large amount of web text. It works on **bytes**, so it can handle any text at all, in any script. A space is treated as part of the token that follows it, and is shown as `Ġ`.

In [ ]:
%pip install -q transformers
from transformers import AutoTokenizer

tokeniser = AutoTokenizer.from_pretrained("gpt2")
print("Vocabulary size:", tokeniser.vocab_size)

for text in ["The cat sat on the mat.", "unbelievable", "Tokenisation is surprisingly interesting.", "Η γλώσσα είναι όμορφη."]:
    pieces = tokeniser.tokenize(text)
    print(f"\n{text}\n   {len(pieces)} tokens: {pieces}")

Three things to notice in the output.

- Common English words are single tokens, and rarer or longer words are split into pieces.
- The last line is Greek. GPT-2's merges were learned almost entirely from English, so Greek text is broken into many small fragments of bytes. The same sentence costs several times more tokens than its English equivalent. A model handles a language well only if its tokeniser was trained with plenty of that language.
- The numbers a model works with are these token ids, never words. `tokeniser.encode(text)` gives the ids and `tokeniser.decode(ids)` turns them back into text.

### Why you should care about tokens
- **Limits.** A model can read only a fixed number of tokens at once, its *context window*.
- **Cost.** Commercial models charge per token, for input and output alike.
- **Odd behaviour.** Tasks that depend on individual letters, such as counting the letters in a word or spelling it backwards, are harder for a model than you would expect, because it sees pieces and not letters.

> ⚠️ **Common pitfalls**
>
> - Estimating length in words. As a rule of thumb, English text has about 4 tokens for every 3 words. Other languages, source code and unusual names can take far more.
> - Using the wrong tokeniser. Every model must be used with **the tokeniser it was trained with**. The ids of one tokeniser are meaningless to another model.
> - Assuming the pieces are meaningful units. They are frequent sequences. Often they coincide with stems and endings, and often they do not.

## 4. From probabilities to text

We now turn to the other end. A language model, of any kind, does one thing: given the text so far, it outputs a **probability for every token in its vocabulary** being the next one. It does not output a word. Turning that distribution into an actual choice, again and again, is called **decoding**, and there are several ways to do it.

To study them we use a small made-up distribution: the possible next words after *"The cat sat on the"*.

In [ ]:
next_words = ["mat", "floor", "sofa", "roof", "table", "moon", "piano", "democracy"]
probabilities = np.array([0.40, 0.20, 0.15, 0.10, 0.08, 0.04, 0.02, 0.01])

rng = np.random.default_rng(0)

def sample(probabilities, how_many=12):
    return [next_words[i] for i in rng.choice(len(next_words), size=how_many, p=probabilities)]

print("Sum of the probabilities:", probabilities.sum())
print("Greedy choice:", next_words[int(np.argmax(probabilities))])
print("Twelve samples:", sample(probabilities))

### Greedy decoding and plain sampling
**Greedy decoding** always takes the most probable token. It is predictable: the same input gives the same output every time. It is also dull, and over a long text it tends to fall into loops, because the most likely continuation of a phrase is often the phrase again.

**Sampling** draws a token at random according to the probabilities, as we did for the n-gram model in Notebook 07 and the LSTM in Notebook 12. It is varied, and it has the opposite fault. The vocabulary of a real model has tens of thousands of tokens, most of them wildly inappropriate at any given point, each with a tiny probability. Together those tiny probabilities add up to a real chance of choosing something absurd. The word *democracy* above stands for that long tail.

The three methods below are all ways of **reshaping** the distribution before sampling from it.

### Temperature
We met temperature in Notebook 12. It is applied to the model's raw scores (the *logits*) before the softmax:

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

A temperature below 1 sharpens the distribution towards its peak. A temperature above 1 flattens it. As $T$ approaches 0 the result approaches greedy decoding.

In [ ]:
def with_temperature(probabilities, temperature):
    logits = np.log(probabilities) / temperature
    reshaped = np.exp(logits - logits.max())
    return reshaped / reshaped.sum()

figure, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
for axis, temperature in zip(axes, [0.5, 1.0, 2.0]):
    axis.bar(next_words, with_temperature(probabilities, temperature))
    axis.set_title(f"temperature {temperature}")
    axis.tick_params(axis="x", rotation=60)
axes[0].set_ylabel("probability")
plt.tight_layout()
plt.show()

for temperature in [0.5, 1.0, 2.0]:
    reshaped = with_temperature(probabilities, temperature)
    print(f"T = {temperature}: P(mat) = {reshaped[0]:.2f}   P(democracy) = {reshaped[-1]:.3f}")

Temperature changes how adventurous the model is, and it has a weakness: it never removes anything. At a high temperature the absurd options become **more** likely along with the interesting ones.

### Top-k sampling
Keep only the $k$ most probable tokens, discard the rest, and rescale what is left so that it adds up to 1.

In [ ]:
def top_k(probabilities, k):
    kept = np.zeros_like(probabilities)
    best = np.argsort(-probabilities)[:k]
    kept[best] = probabilities[best]
    return kept / kept.sum()

print({word: round(float(p), 2) for word, p in zip(next_words, top_k(probabilities, 3)) if p > 0})
print("Twelve samples:", sample(top_k(probabilities, 3)))

The absurd tail is gone. The difficulty is choosing $k$. After *"The capital of France is"* there is one sensible continuation, and after *"My favourite food is"* there are hundreds. A fixed $k$ is too generous in the first case and too strict in the second.

### Top-p (nucleus) sampling
Top-p lets the size of the shortlist **adapt**. Sort the tokens from most to least probable, and keep the smallest set whose probabilities add up to at least $p$. When the model is confident, that set is one or two tokens. When it is unsure, it is many.

In [ ]:
def top_p(probabilities, p):
    order = np.argsort(-probabilities)
    cumulative = np.cumsum(probabilities[order])
    how_many = int(np.searchsorted(cumulative, p - 1e-9) + 1)      # smallest set that reaches p
    kept = np.zeros_like(probabilities)
    kept[order[:how_many]] = probabilities[order[:how_many]]
    return kept / kept.sum()

for p in [0.5, 0.75, 0.9]:
    reshaped = top_p(probabilities, p)
    print(f"top-p {p}: keeps {[word for word, q in zip(next_words, reshaped) if q > 0]}")

confident = np.array([0.93, 0.03, 0.01, 0.01, 0.01, 0.005, 0.003, 0.002])
print("\nA confident distribution, top-p 0.9:", [word for word, q in zip(next_words, top_p(confident, 0.9)) if q > 0])

With the same setting of 0.9, the shortlist has five words when the model is unsure and a single word when it is confident. That adaptability is why top-p, usually combined with a temperature, is the default in most systems that generate text today.

## 5. Searching for the best sequence: beam search

### Intuition first
Sampling is about variety. Sometimes we want the opposite: the single **most probable** output. For a translation or a summary there is, roughly, a best answer.

Greedy decoding does not find it. The best first word does not necessarily begin the best **sentence**. A slightly less likely first word may open the way to much more likely continuations, like a chess move that looks second-best and wins the game.

Checking every possible sequence is impossible: with 50,000 tokens there are $50{,}000^{20}$ sequences of length 20. **Beam search** is the practical middle course. Instead of following one path, follow the best few.

### Formal definition
With a **beam width** $B$:

1. Keep the $B$ best partial sequences found so far (the *beam*).
2. Extend each of them by every possible next token, and score each extension by the sum of the log-probabilities of its tokens.
3. Keep the best $B$ of all the extensions and discard the rest.
4. Repeat.

Greedy decoding is beam search with $B = 1$. To try it we need a language model with real probabilities, so we rebuild the bigram model of Notebook 07.

In [ ]:
bigram = {}
for sentence in gutenberg.sents("austen-emma.txt"):
    tokens = ["<s>"] + [word.lower() for word in sentence if word.isalpha()] + ["</s>"]
    for first, second in zip(tokens, tokens[1:]):
        if first not in bigram:
            bigram[first] = Counter()
        bigram[first][second] += 1

def beam_search(start, length, width):
    beam = [([start], 0.0)]                                    # (words so far, log-probability)
    for step in range(length):
        candidates = []
        for words, score in beam:
            counts = bigram[words[-1]]
            total = sum(counts.values())
            for word, count in counts.items():
                if word != "</s>":
                    candidates.append((words + [word], score + math.log(count / total)))
        beam = sorted(candidates, key=lambda candidate: candidate[1], reverse=True)[:width]
    return beam

for width in [1, 3, 10]:
    words, score = beam_search("she", length=8, width=width)[0]
    print(f"beam width {width:>2}: log-probability {score:7.2f}   {' '.join(words)}")

The log-probability rises (towards zero) as the beam widens: beam search has found sequences that the bigram model considers more probable than the greedy one. Greedy decoding was **not** optimal.

Now read the sentences. The most probable sequence is not the most interesting one. It is made of the most common words in the most common order. This is a general and somewhat surprising fact about language models: **the most likely text is bland**, because safe, generic phrases are probable everywhere. Real human text is less probable than the text a model would choose by maximising.

So the choice of decoding method follows from the task.

| Task | What we want | Typical choice |
|---|---|---|
| translation, summarisation, speech recognition | the one most accurate output | beam search, or greedy |
| answering factual questions, extracting data, writing code | reliability and repeatability | greedy, or a low temperature |
| stories, brainstorming, conversation | variety and surprise | sampling with top-p and a moderate temperature |

## 6. Decoding with a real model

*This section downloads GPT-2 (about 500 MB) and is meant for Colab.*

GPT-2 is a small ancestor of today's large models, released in 2019. We give it the same beginning each time and change only the decoding settings.

In [ ]:
from transformers import pipeline, set_seed

generator = pipeline("text-generation", model="gpt2")
prompt = "The old lighthouse keeper opened the door and"

def show(title, **settings):
    set_seed(0)
    text = generator(prompt, max_new_tokens=40, pad_token_id=50256, **settings)[0]["generated_text"]
    print(f"--- {title}\n{text}\n")

show("greedy", do_sample=False)
show("beam search, width 5", do_sample=False, num_beams=5)
show("sampling, temperature 0.7", do_sample=True, temperature=0.7, top_k=0)
show("sampling, temperature 1.5", do_sample=True, temperature=1.5, top_k=0)
show("top-k 40", do_sample=True, top_k=40)
show("top-p 0.9", do_sample=True, top_p=0.9, top_k=0)

Compare what you see with what Sections 4 and 5 predicted. Greedy and beam search tend to produce correct, flat text, and often repeat a phrase. A high temperature without any cut-off drifts into incoherence. Top-k and top-p are varied and remain readable.

It is the **same model with the same weights** in every case. Much of what people perceive as the "personality" of a text generator, cautious or creative, repetitive or erratic, is decided by a few lines of decoding code that run after the network has finished.

---
## ✏️ Exercises

### Exercise 1 (BPE on paper)
Take the single "word" `aaabdaaabac` (ignore the end-of-word marker for this exercise). On paper, perform three BPE merges: at each step find the most frequent pair of neighbouring symbols and merge it. Then check with the code below, which calls `learn_bpe` and shows each step.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
example_merges = learn_bpe(Counter(["aaabdaaabac"]), 3, show=True)
```

*The pair "aa" occurs most often and is merged first, then that new symbol with the next letter, and so on. After three merges the string of 11 letters is written with far fewer symbols. This is the data-compression algorithm that BPE originally was. (The code adds an end-of-word marker, which does not change the first merges.)*
</details>

### Exercise 2 (How many merges?)
A useful property of BPE: the first $n$ merges of a longer list are exactly the tokeniser you would get by learning only $n$ merges. Use it to compare tokenisers with 0, 100, 500 and 1,000 merges, by slicing the list `merges`. For each, compute the average number of tokens per word when tokenising *Alice* (`alice_words`). What are the two extremes?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
alice_counts = Counter(alice_words)

for number in [0, 100, 500, 1000]:
    rules = merges[:number]
    total = 0
    for word, count in alice_counts.items():
        total += len(bpe_tokenise(word, rules)) * count
    print(f"{number:>5} merges: {total / len(alice_words):.2f} tokens per word")
```

*With no merges every word is spelled out letter by letter, plus the end marker: a character-level tokeniser. As merges are added the number falls towards 1, a word-level tokeniser. The number of merges is a dial between the two.*
</details>

### Exercise 3 (How random is it?)
The **entropy** of a distribution, $H = -\sum_i p_i \log_2 p_i$, measures its uncertainty in bits. Compute the entropy of `probabilities` after applying temperatures 0.25, 0.5, 1, 2 and 4. What are the smallest and largest values entropy could possibly take for 8 options?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def entropy(distribution):
    positive = distribution[distribution > 0]
    return float(-(positive * np.log2(positive)).sum())

for temperature in [0.25, 0.5, 1, 2, 4]:
    print(f"T = {temperature:<5} entropy {entropy(with_temperature(probabilities, temperature)):.2f} bits")
```

*Entropy rises with temperature. The minimum is 0 bits, when one option has all the probability (greedy decoding). The maximum for 8 options is 3 bits, when all are equally likely, since log2(8) = 3. Perplexity, from Notebook 07, is simply 2 raised to the entropy.*
</details>

### Exercise 4 (Combine the methods)
Real systems apply a temperature **and then** top-p. Write `reshape(probabilities, temperature, p)` that does both, in that order. Using temperature 1.5, compare the words that survive with top-p 0.9 against those that survive at temperature 1.0. Why does the order of the two steps matter?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def reshape(probabilities, temperature, p):
    return top_p(with_temperature(probabilities, temperature), p)

for temperature in [1.0, 1.5]:
    result = reshape(probabilities, temperature, 0.9)
    print(f"temperature {temperature}:", {word: round(float(q), 2) for word, q in zip(next_words, result) if q > 0})
```

*A higher temperature flattens the distribution first, so more words are needed to reach 90% and the shortlist grows. Applied in the other order, the cut would be made on the original distribution and the temperature would only rearrange the survivors. The two settings interact, which is why changing both at once makes behaviour hard to predict.*
</details>

### Exercise 5 (Wider beams)
Run `beam_search` from the word `"i"` and from the word `"he"`, for 10 words, with beam widths 1, 2, 5, 20 and 100. Print the log-probability and the sentence each time. Does a wider beam always give a higher score? Does the text become better?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for start in ["i", "he"]:
    for width in [1, 2, 5, 20, 100]:
        words, score = beam_search(start, length=10, width=width)[0]
        print(f"width {width:>3}: {score:7.2f}   {' '.join(words)}")
    print()
```

*The score can only stay the same or improve as the beam widens. From "he" it improves step by step. From "i" every width gives the same answer, so the greedy choice was already the best one, and that best sequence is a loop: "i am sure i am sure". The text does not get better as the score rises. Wide beams settle on the most common words and phrases. Higher probability and higher quality are different things.*
</details>

### Exercise 6 (Counting tokens)
This exercise uses the GPT-2 `tokeniser` of Section 3, so run it in Colab. Count the tokens in the same sentence written in English and in two other languages you know. Then compute the number of tokens per character for each.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
sentences = {
    "English": "Good morning, how are you today?",
    "Greek":   "Καλημέρα, τι κάνεις σήμερα;",
    "French":  "Bonjour, comment allez-vous aujourd'hui ?",
}
for language, sentence in sentences.items():
    count = len(tokeniser.encode(sentence))
    print(f"{language:<8} {count:>3} tokens   {count / len(sentence):.2f} tokens per character")
```

*The same greeting costs several times more tokens in a language the tokeniser was not built for. With a model that charges per token and has a fixed context window, speakers of such languages pay more and fit less. Newer multilingual tokenisers narrow this gap, and do not close it completely.*
</details>

## 🔑 Key takeaways

- **Words** as units leave rare words unknown. **Characters** make sequences long. **Subwords** keep frequent words whole and split rare ones into reusable pieces.
- **BPE** learns the pieces by repeatedly merging the most frequent pair of neighbouring symbols. The ordered list of merges is the tokeniser.
- Models read and write **token ids**. Tokens determine cost and context limits, and each model needs its own tokeniser.
- A language model outputs a **distribution**. **Decoding** turns it into text.
- **Greedy** decoding is repeatable and dull. **Temperature** sharpens or flattens the distribution. **Top-k** and **top-p** cut off the unreliable tail, and top-p adapts to the model's confidence.
- **Beam search** looks for the most probable sequence, and the most probable text is bland.
- Choose the method for the task: accuracy calls for beam search or low temperature, creativity for sampling.

---
**Next:** *Notebook 16: Text from the Web*, where we look at where the text that all these models learn from comes from, and how to collect it properly.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*